## postcode price predictor

interactive front end for `src/predict.py` -- same model and features as `02_modelling.ipynb`'s serving subset, just packaged for a single "what would this property sell for" query instead of evaluating the model as a whole.

requires `python -m src.predict --train` to have been run at least once (already done -- it saved a trained model, two quantile models for an 80% interval, and snapshots of each area's recent price data to `data/processed/`). re-run `--train` only if the underlying sales data changes.

inputs:
- **postcode** -- any UK postcode, quoted
- **property_type** -- `F` flat, `S` semi-detached, `T` terraced, `D` detached
- **duration** -- `F` freehold, `L` leasehold
- **old_new** -- optional, defaults to `N` (existing property), or `Y` for new build

returns `(point, lower, upper)` -- a point estimate plus an 80% prediction interval, not just a single number (see below for why).

## what this prediction is (and isn't) based on

this model only knows what's in land registry sale records + postcode coordinates:

- property type, tenure (freehold/leasehold), new-build flag
- distance from central london
- the area's own recent sale prices/volume at two hex resolutions, its immediate neighbours' prices, and a longer-run price-tier estimate (the hex features -- see `01_eda.ipynb`/`02_modelling.ipynb`)

it does **not** observe, and cannot separately account for:

- anything about the *specific* property -- condition, size, bedroom count, garden, parking, renovation state, exact street/address. (street-level features exist in the research model in `02_modelling.ipynb` and measurably help there, but need a specific street/house a postcode-only query doesn't have, so they're not part of what this tool serves)
- crime rate, school catchment, transit links, green space, or any other named local amenity -- none of these are measured directly. they only show up *indirectly*, to the extent they are already baked into an area's recent sale prices (the hex trailing price is a proxy for "whatever locally drives value here," not a list of named factors -- see the hex-price assumption in the README)

scope limits:

- trained only on greater london sales, 2023-01 to 2025-12 -- a postcode outside that radius raises an error rather than extrapolating (demonstrated below)
- "now" for this model means the end of the training window (2025-12-31), not today's actual market -- prices are not refreshed in real time
- the point estimate comes from a model that explains roughly 64% of price variation (R² in `02_modelling.ipynb`'s serving-feature-set evaluation) -- the 80% interval is measured to actually cover about 78% of true prices in cross-validation (close to nominal, not just a decorative range), and it's often wide (commonly closer to double the lower bound) -- read both as a ballpark for a *type* of property in an *area*, not a valuation of one specific house
- `old_new` silently defaults to `N` (existing property) if you don't pass it
- independently-trained quantile models have no ordering guarantee against each other or the point estimate -- `predict_price` clips the interval so it always contains the point estimate, rather than ever showing a nonsensical crossed range

In [1]:
import sys

sys.path.append("..")
from src.predict import predict_price

/Users/cam/Documents/datascience/cv_projects/house_price_prediction/house_price_prediction_venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
point, lower, upper = predict_price("SW4 6BZ", "F", "L")
f"£{point:,.0f} (80% interval: £{lower:,.0f} - £{upper:,.0f})"

'£537,013 (80% interval: £365,947 - £797,241)'

comparing locations only makes sense holding property type and tenure constant -- a flat in one postcode against a house in another mixes the location effect with the type effect and the comparison stops meaning anything.

In [3]:
for postcode in ["SW4 6BZ", "CR5 2JD"]:
    point, lower, upper = predict_price(postcode, "T", "F")
    print(postcode, "->", f"£{point:,.0f} (80% interval: £{lower:,.0f} - £{upper:,.0f})")

SW4 6BZ -> £1,015,373 (80% interval: £682,868 - £1,542,888)


CR5 2JD -> £490,091 (80% interval: £399,557 - £633,836)


postcodes too far outside greater london raise an error rather than silently returning a number -- the model was never trained on anything past ~32km from central london.

In [4]:
try:
    predict_price("M1 1AE", "F", "L")
except ValueError as e:
    print("error:", e)

error: postcode 'M1 1AE' is outside the trained region (too far from central london)
